# 11f — Dipole diagnostic: mag(psfFlux) − mag(apFlux) per band

## Scientific motivation

Atmospheric dispersion (index of refraction of the air) shifts stellar images in
the North–South direction as a function of wavelength.  When the Rubin AP pipeline
tries to subtract a template coadd from a science image, the slight positional
mismatch between the PSF centroid and the aperture centre creates a **dipole**
residual in the East–West direction at second order.

One observable signature of this effect is a **systematic difference between the
PSF-fit flux and the aperture flux** on the difference image:

- `psfFlux` (nJy): PSF-weighted flux on the difference image — sensitive to
  centroid shifts because the PSF model is centred on the science-image star.
- `apFlux` (nJy): fixed circular aperture flux on the difference image — less
  sensitive to centroid shifts but integrates over the dipole lobes symmetrically.

For a perfect subtraction with no centroid shift, `psfFlux ≈ apFlux` and
`mag(psfFlux) − mag(apFlux) ≈ 0`.
A **systematic non-zero offset** or a **larger scatter** for dipole-flagged
sources would confirm that the PSF/aperture discrepancy is driven by dipoles.

## Contents

1. **Figure 1** — 2×3 subplots, one per band:  
   histogram of `Δmag = mag(psfFlux) − mag(apFlux)` per visit,  
   grey = non-dipole sources, band colour = dipole-flagged sources.

2. **Figure 2** — all bands combined in one panel:  
   grey histogram = all non-dipole sources,  
   coloured histograms (one per band, using band colour) = dipole sources.

3. **Figure 3** — 2×3 subplots, one per band — **stacked by category**:  
   Each subplot shows two stacked bar-histograms side by side:  
   - **bottom half** of each bar: non-dipole counts per category (lighter shade)  
   - **top half** of each bar: dipole counts per category (saturated shade)  
   The three categories are colour-coded (`steelblue`, `seagreen`, `firebrick`).  
   This allows a direct visual comparison of whether the Δmag distribution
   **shape or median** differs between Gaia stable HQ, Gaia stable no-phot,
   and Gaia variable stars — both for dipole and non-dipole populations.

**Stellar categories analysed:**
- `gaia_star_stable_hq` — Gaia photometric stable HQ stars
- `gaia_nophotgstar_stable_unknown_parallax` — Gaia stable, no photometric solution
- `gaia_star_variable` — Gaia variable stars (control group)

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS, Université Paris-Saclay)  
**Creation Date:** 2026-05-22  
**Notebook tag:** 11f

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

warnings.filterwarnings("ignore")

print(f"pandas     : {pd.__version__}")
print(f"numpy      : {np.__version__}")
print(f"matplotlib : {mpl.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl → interactive backend")
except ImportError:
    %matplotlib inline
    print("ipympl not found → inline backend")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input parquets from notebook 01
DIR_FIGS = "figs_FINK_BLOCK_LC_11f"  # output figures
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Photometric bands ──────────────────────────────────────────────────────────
BANDS = list("ugrizy")
BANDS_ROW0 = list("ugr")
BANDS_ROW1 = list("izy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Stellar categories ─────────────────────────────────────────────────────────
CATEGORIES = {
    "gaia_star_stable_hq": {
        "label": "Gaia stable HQ",
        "color": "firebrick",
    },
    "gaia_nophotgstar_stable_unknown_parallax": {
        "label": "Gaia stable no-phot",
        "color": "seagreen",
    },
    "gaia_star_variable": {
        "label": "Gaia variable",
        "color": "steelblue",
    },
}

# ── AB magnitude zero-point ────────────────────────────────────────────────────
# m_AB = -2.5 * log10(f_nJy / AB_FLUX_ZERO)
AB_FLUX_ZERO = 3631e9  # nJy

# ── Histogram binning for Δmag ────────────────────────────────────────────────
# mag(psfFlux) - mag(apFlux) is expected to be small and centred near 0.
# Range [-3, +3] with 0.1 mag bins captures both the bulk distribution
# and the extended dipole tails.
DMAG_BINS = np.arange(-3.0, 3.01, 0.1)
DMAG_RANGE = (-3.0, 3.0)  # used for combined figure axis

# ── Plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.30,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    """Save the current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        fpath = os.path.join(DIR_FIGS, f"{name}.{ext}")
        plt.savefig(fpath, bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print(f"Data dir : {os.path.abspath(DIR_DATA)}")
print(f"Figs dir : {os.path.abspath(DIR_FIGS)}")

## 2. Utility functions

In [ ]:
def flux_nJy_to_mag_AB(flux_nJy: np.ndarray) -> np.ndarray:
    """Convert flux in nJy to AB magnitude.

    Returns NaN for non-positive or non-finite flux values.

    Parameters
    ----------
    flux_nJy : array-like  flux in nano-Jansky

    Returns
    -------
    np.ndarray  AB magnitudes (NaN where input is ≤ 0)
    """
    f = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(f > 0, -2.5 * np.log10(f / AB_FLUX_ZERO), np.nan)


def parse_dipole_bool(series: pd.Series) -> pd.Series:
    """Coerce a dipole-flag column (bool / int / str) to a boolean Series."""

    def _cast(val):
        if isinstance(val, bool):
            return val
        if isinstance(val, (int, float)):
            return bool(val)
        if isinstance(val, str):
            return val.strip().lower() in ("true", "1", "yes")
        return False

    return series.apply(_cast)


def compute_dmag_psf_ap(df: pd.DataFrame) -> pd.Series:
    """Compute mag(psfFlux) - mag(apFlux) for each row.

    Both fluxes must be positive for the magnitude difference to be defined.
    Rows with non-positive psfFlux or apFlux yield NaN.

    Parameters
    ----------
    df : DataFrame with columns r:psfFlux and r:apFlux in nJy

    Returns
    -------
    pd.Series  Δmag = mag(psfFlux) − mag(apFlux)
    """
    mag_psf = flux_nJy_to_mag_AB(df["r:psfFlux"].values)
    mag_ap = flux_nJy_to_mag_AB(df["r:apFlux"].values)
    return pd.Series(mag_psf - mag_ap, index=df.index, name="dmag_psf_ap")


def lighten_color(color: str, amount: float = 0.50) -> tuple:
    """Lighten a named or hex matplotlib color by blending it towards white.

    Parameters
    ----------
    color  : any matplotlib color spec
    amount : 0 = original, 1 = white

    Returns
    -------
    tuple  RGBA
    """
    import matplotlib.colors as mc

    rgba = np.array(mc.to_rgba(color))
    rgba[:3] = 1.0 - amount * (1.0 - rgba[:3])  # blend RGB towards white
    return tuple(rgba)


print("Utility functions defined.")

## 3. Load parquet data

For each stellar category we load `*_src.parquet` (DIA detections).
Required columns: `r:psfFlux`, `r:apFlux`, `r:isDipole`, `r:band`.

The quantity of interest is:
$$\Delta m = m(\mathrm{psfFlux}) - m(\mathrm{apFlux})
= -2.5 \log_{10}\!\left(\frac{F_{\mathrm{psf}}}{F_{\mathrm{ap}}}\right)$$

This is defined only when both fluxes are positive (i.e. the difference-image
residual has been detected above zero in both estimators).
Rows with non-positive flux are silently dropped from the histograms.

In [ ]:
data = {}  # data[cat] = cleaned DataFrame for that category

for cat in CATEGORIES:
    fpath = os.path.join(DIR_DATA, f"{cat}_src.parquet")
    if not os.path.exists(fpath):
        print(f"[SKIP] {cat}: file not found ({fpath})")
        continue

    df = pd.read_parquet(fpath)

    # ── Mandatory columns check ────────────────────────────────────────────
    required = ["r:psfFlux", "r:apFlux", "r:band"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"[WARN] {cat}: missing columns {missing} — skipped")
        continue

    # ── Parse dipole flag ──────────────────────────────────────────────────
    if "r:isDipole" in df.columns:
        df["is_dipole"] = parse_dipole_bool(df["r:isDipole"].fillna(False))
    else:
        print(f"[WARN] {cat}: r:isDipole absent — setting all to False")
        df["is_dipole"] = False

    # ── Compute Δmag = mag(psfFlux) − mag(apFlux) ─────────────────────────
    df["dmag_psf_ap"] = compute_dmag_psf_ap(df)

    # ── Store ──────────────────────────────────────────────────────────────
    data[cat] = df

    lbl = CATEGORIES[cat]["label"]
    n_tot = len(df)
    n_dip = int(df["is_dipole"].sum())
    n_ok = int(df["dmag_psf_ap"].notna().sum())
    print(
        f"  {lbl:40s}  {n_tot:6d} rows  {n_dip:5d} dipoles ({100 * n_dip / n_tot:.1f}%)  {n_ok:5d} valid Δmag"
    )

CATS_OK = list(data.keys())
print(f"\nCategories loaded: {CATS_OK}")

## 4. Quick inspection of Δmag distributions

In [ ]:
# Per-category descriptive statistics of Δmag split by dipole flag
rows_stat = []
for cat in CATS_OK:
    df = data[cat]
    lbl = CATEGORIES[cat]["label"]
    for is_dip, tag in [(False, "non-dipole"), (True, "dipole")]:
        sub = df[df["is_dipole"] == is_dip]["dmag_psf_ap"].dropna()
        rows_stat.append(
            {
                "category": lbl,
                "type": tag,
                "N": len(sub),
                "median": float(np.nanmedian(sub)) if len(sub) else np.nan,
                "mean": float(np.nanmean(sub)) if len(sub) else np.nan,
                "std": float(np.nanstd(sub)) if len(sub) else np.nan,
                "p16": float(np.nanpercentile(sub, 16)) if len(sub) else np.nan,
                "p84": float(np.nanpercentile(sub, 84)) if len(sub) else np.nan,
            }
        )

df_stat = pd.DataFrame(rows_stat)
print("Descriptive statistics of  Δmag = mag(psfFlux) − mag(apFlux):")
display(
    df_stat.style.format(
        {"median": "{:.4f}", "mean": "{:.4f}", "std": "{:.4f}", "p16": "{:.4f}", "p84": "{:.4f}"}
    )
)

## 5. Figure 1 — 2×3 subplots: Δmag histogram per band

One subplot per photometric band (layout: u g r / i z y).  
For each subplot:
- **Grey filled histogram** — sources **without** dipole flag (`isDipole = False`)
- **Band-colour filled histogram** — sources **with** dipole flag (`isDipole = True`)

The two histograms are overlaid with transparency so the overlap is visible.
Both are normalised to **density** so their shapes are directly comparable
regardless of the very different sample sizes.

All stellar categories are **merged** for these figures, with the category origin
encoded in `gaia_origin` for reference.

In [ ]:
# Build a single merged DataFrame for all categories
frames = []
for cat in CATS_OK:
    tmp = data[cat].copy()
    tmp["gaia_origin"] = cat
    tmp["cat_label"] = CATEGORIES[cat]["label"]
    frames.append(tmp)

df_all = pd.concat(frames, ignore_index=True)
print(f"Total rows (all categories merged): {len(df_all):,}")
print(
    f"Total dipoles                      : {int(df_all['is_dipole'].sum()):,}  "
    f"({100 * df_all['is_dipole'].mean():.1f}%)"
)
print(f"Valid Δmag rows                    : {int(df_all['dmag_psf_ap'].notna().sum()):,}")

In [ ]:
def plot_fig1_per_band(df, bins=DMAG_BINS, subtitle=None, savename=None):
    """Figure 1: 2×3 subplots — one per band.

    Each subplot shows overlaid histograms of
    Δmag = mag(psfFlux) − mag(apFlux):
      - grey   : non-dipole sources
      - colour : dipole-flagged sources (band colour)

    Histograms are normalised to probability density so shapes are
    directly comparable across populations of different sizes.

    Parameters
    ----------
    df       : DataFrame (one or all categories)
    bins     : bin edges for Δmag histogram
    subtitle : optional string appended to suptitle
    savename : if given, save PDF + PNG to DIR_FIGS
    """
    fig, axes = plt.subplots(2, 3, figsize=(5.5 * 3, 4.5 * 2), squeeze=False)

    extra = f"\n{subtitle}" if subtitle else "\nAll stellar categories merged"
    fig.suptitle(
        r"$\Delta m = m(\mathrm{psfFlux}) - m(\mathrm{apFlux})$ per band"
        + extra
        + r" — $\blacksquare$ grey = no dipole   $\blacksquare$ colour = dipole",
        fontsize=12,
        fontweight="bold",
        y=1.02,
    )

    hist_kw = dict(bins=bins, density=True, histtype="stepfilled", edgecolor="white", linewidth=0.4)

    for row_idx, bands_row in enumerate([BANDS_ROW0, BANDS_ROW1]):
        for col_idx, band in enumerate(bands_row):
            ax = axes[row_idx][col_idx]
            bcolor = BAND_COLORS[band]
            df_b = df[df["r:band"] == band]

            dmag_nodip = df_b.loc[~df_b["is_dipole"], "dmag_psf_ap"].dropna().values
            dmag_dip = df_b.loc[df_b["is_dipole"], "dmag_psf_ap"].dropna().values

            if len(dmag_nodip) > 0:
                ax.hist(
                    dmag_nodip,
                    color="#888888",
                    alpha=0.55,
                    label=f"no dipole  (N={len(dmag_nodip)})",
                    **hist_kw,
                )
                ax.axvline(
                    np.median(dmag_nodip),
                    color="#555555",
                    lw=1.2,
                    ls=":",
                    alpha=0.80,
                    label=f"med no-dip = {np.median(dmag_nodip):.3f}",
                )

            if len(dmag_dip) > 0:
                ax.hist(
                    dmag_dip, color=bcolor, alpha=0.75, label=f"dipole      (N={len(dmag_dip)})", **hist_kw
                )
                ax.axvline(
                    np.median(dmag_dip),
                    color=bcolor,
                    lw=1.4,
                    ls=":",
                    alpha=0.95,
                    label=f"med dipole = {np.median(dmag_dip):.3f}",
                )

            ax.axvline(0.0, color="black", lw=1.0, ls="--", alpha=0.60, label=r"$\Delta m = 0$", zorder=5)
            ax.set_xlabel(r"$m(\mathrm{psfFlux}) - m(\mathrm{apFlux})$  (mag)", fontsize=9)
            ax.set_ylabel("Probability density", fontsize=9)
            ax.set_title(f"band  {band}", color=bcolor, fontweight="bold", fontsize=11)
            ax.set_xlim(DMAG_RANGE)
            ax.legend(fontsize=7, loc="upper right", framealpha=0.75)

    plt.tight_layout()
    if savename:
        savefig(savename)
    plt.show()


print("plot_fig1_per_band() defined.")

In [ ]:
plot_fig1_per_band(df_all, savename="11f_fig1_dmag_psfAp_per_band")

## 6. Figure 2 — All bands combined in one panel

A single axis shows:
- **Grey histogram** — all non-dipole sources across all bands combined
- **Coloured step histograms** (one per band, using `BAND_COLORS`) —
  dipole-flagged sources for each band

This figure reveals whether the PSF–aperture flux discrepancy for dipoles is
band-dependent (e.g. larger in bluer bands where atmospheric dispersion is
stronger) or roughly uniform across bands.

In [ ]:
def plot_fig2_all_bands(df, bins=DMAG_BINS, savename=None):
    """Figure 2: all bands combined in a single panel.

    - Grey stepfilled histogram : all non-dipole sources (all bands)
    - Coloured step histograms  : dipole sources per band (band colour)

    Both are normalised to probability density.
    """
    fig, ax = plt.subplots(figsize=(9, 5))

    dmag_nodip_all = df.loc[~df["is_dipole"], "dmag_psf_ap"].dropna().values
    if len(dmag_nodip_all) > 0:
        ax.hist(
            dmag_nodip_all,
            bins=bins,
            density=True,
            histtype="stepfilled",
            color="#888888",
            alpha=0.45,
            edgecolor="white",
            linewidth=0.5,
            label=f"all bands, no dipole  (N={len(dmag_nodip_all)})",
            zorder=2,
        )
        med_nd = np.median(dmag_nodip_all)
        ax.axvline(
            med_nd,
            color="#444444",
            lw=1.3,
            ls=":",
            alpha=0.85,
            zorder=6,
            label=f"median no-dipole = {med_nd:.3f}",
        )

    for band in BANDS:
        bcolor = BAND_COLORS[band]
        dmag_dip_b = df.loc[df["is_dipole"] & (df["r:band"] == band), "dmag_psf_ap"].dropna().values
        if len(dmag_dip_b) == 0:
            continue
        ax.hist(
            dmag_dip_b,
            bins=bins,
            density=True,
            histtype="step",
            color=bcolor,
            lw=2.0,
            alpha=0.90,
            zorder=4,
            label=f"band {band}, dipole  (N={len(dmag_dip_b)})",
        )
        ax.axvline(np.median(dmag_dip_b), color=bcolor, lw=1.2, ls="--", alpha=0.80, zorder=5)

    ax.axvline(0.0, color="black", lw=1.2, ls="-", alpha=0.60, zorder=7, label=r"$\Delta m = 0$")
    ax.set_xlabel(r"$m(\mathrm{psfFlux}) - m(\mathrm{apFlux})$  (mag)", fontsize=11)
    ax.set_ylabel("Probability density", fontsize=11)
    ax.set_title(
        r"$\Delta m = m(\mathrm{psfFlux}) - m(\mathrm{apFlux})$ — all bands"
        "\nGrey = no dipole (all bands)   Coloured step = dipole per band",
        fontsize=11,
        fontweight="bold",
    )
    ax.set_xlim(DMAG_RANGE)
    ax.legend(fontsize=8, loc="upper right", framealpha=0.80, ncol=2)

    plt.tight_layout()
    if savename:
        savefig(savename)
    plt.show()


print("plot_fig2_all_bands() defined.")

In [ ]:
plot_fig2_all_bands(df_all, savename="11f_fig2_dmag_psfAp_all_bands")

## 7. Figure 3 — 2×3 subplots: Δmag stacked by category (no-dipole ↓ / dipole ↑)

### Goal

Directly compare whether the `Δmag = mag(psfFlux) − mag(apFlux)` distribution
**shape or median** differs between the three Gaia categories, both for dipole
and non-dipole populations.

### Layout trick: vertical stacking within each bin

Each Δmag bin contains a **stacked bar** split into two halves:

- **Bottom half** (positive y, lighter shade): non-dipole counts per category.  
  The three categories are stacked on top of each other using
  `matplotlib ax.bar(..., bottom=...)`, so their relative contributions to
  the non-dipole population are immediately visible.

- **Top half** (positive y, saturated shade, starts where the bottom half ends):
  dipole counts per category, stacked in the same order.

A **horizontal black dashed line** separates the two halves at the boundary
between the total non-dipole stack and the first dipole layer.

### Colour convention

| Category | Saturated (dipole) | Light (no-dipole) |
|---|---|---|
| Gaia stable HQ | `steelblue` | `steelblue` 50% lightened |
| Gaia stable no-phot | `seagreen` | `seagreen` 50% lightened |
| Gaia variable | `firebrick` | `firebrick` 50% lightened |

The same colour code is used in both halves so the eye can track each category
from bottom to top.  The lighter shade immediately signals "non-dipole"; the
saturated shade signals "dipole".

### Reading the figure

- If the Δmag **shape** is the same for all categories, all three coloured
  segments will track each other proportionally across bins.
- A **shift in the centre of mass** of one category's stack (e.g. firebrick
  consistently offset to positive Δmag) would indicate that Gaia variable stars
  have a systematically different PSF–aperture flux relation.
- A disproportionately **tall dipole segment** (saturated colour on top) in a
  specific Δmag range would flag that dipoles from that category cluster at a
  particular flux-ratio value.

In [ ]:
def plot_fig3_stacked_categories(data_dict, cats, bins=DMAG_BINS, savename=None):
    """Figure 3: 2×3 subplots — one per band — stacked histogram by category.

    Within each subplot the Δmag bins are split into two vertical sections:
      - Bottom stack (lighter shades): non-dipole counts for each category
      - Top    stack (saturated shades): dipole counts for each category

    The stacking is done with matplotlib ax.bar(..., bottom=...) so that:
      * category contributions within each dipole-class are additive
      * the boundary between the non-dipole and dipole stacks is marked
        by a horizontal dashed line drawn at the top of the non-dipole stack

    All counts are *raw* (not density-normalised) so the absolute number of
    sources per category is directly readable from bar height.

    Parameters
    ----------
    data_dict : dict  {cat_key: DataFrame}  — one entry per category
    cats      : list  ordered list of category keys to stack
    bins      : array  Δmag bin edges
    savename  : str|None  save PDF+PNG if given
    """
    bin_centres = 0.5 * (bins[:-1] + bins[1:])
    bin_width = np.diff(bins)
    bar_width = bin_width * 0.85  # slight gap between bars

    # Pre-compute counts: counts[cat][is_dip] = 1-D array (n_bins,)
    counts = {}
    for cat in cats:
        if cat not in data_dict:
            continue
        df = data_dict[cat]
        counts[cat] = {}
        for band in BANDS:
            df_b = df[df["r:band"] == band]
            dmag_nd = df_b.loc[~df_b["is_dipole"], "dmag_psf_ap"].dropna().values
            dmag_d = df_b.loc[df_b["is_dipole"], "dmag_psf_ap"].dropna().values
            n_nd, _ = np.histogram(dmag_nd, bins=bins)
            n_d, _ = np.histogram(dmag_d, bins=bins)
            counts[cat][band] = {"nodip": n_nd.astype(float), "dip": n_d.astype(float)}

    cats_present = [c for c in cats if c in counts]

    # ── Build figure ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(5.5 * 3, 5.0 * 2), squeeze=False)
    fig.suptitle(
        r"$\Delta m = m(\mathrm{psfFlux}) - m(\mathrm{apFlux})$"
        " — stacked by category\n"
        "Light shade = no dipole (bottom)   "
        "Saturated shade = dipole (top)   "
        "Dashed line = no-dipole / dipole boundary",
        fontsize=12,
        fontweight="bold",
        y=1.02,
    )

    for row_idx, bands_row in enumerate([BANDS_ROW0, BANDS_ROW1]):
        for col_idx, band in enumerate(bands_row):
            ax = axes[row_idx][col_idx]
            bcolor = BAND_COLORS[band]

            # Accumulators for stacking bottoms
            bottom_nd = np.zeros(len(bin_centres))  # bottom of non-dipole stack
            bottom_d = np.zeros(len(bin_centres))  # bottom of dipole stack
            # (starts after all no-dipole bars)

            # ── Pass 1: draw all no-dipole bars (lighter shades) ──────────
            for cat in cats_present:
                cinfo = CATEGORIES[cat]
                c_sat = cinfo["color"]  # saturated colour
                c_lite = lighten_color(c_sat, 0.55)  # lightened colour
                n_nd = counts[cat][band]["nodip"]

                ax.bar(
                    bin_centres,
                    n_nd,
                    width=bar_width,
                    bottom=bottom_nd,
                    color=c_lite,
                    edgecolor="white",
                    linewidth=0.3,
                    alpha=0.90,
                    zorder=2,
                    # label only once (first band) to avoid legend duplication
                    label=(f"{cinfo['label']} no-dip" if (row_idx == 0 and col_idx == 0) else None),
                )
                bottom_nd += n_nd

            # bottom_nd now holds the top of the full non-dipole stack per bin
            # → dipole bars start from here
            bottom_d[:] = bottom_nd

            # ── Horizontal separator: top of no-dipole stack ──────────────
            # Draw a step-line at the top of the non-dipole stacks so the eye
            # can immediately locate the boundary between the two populations.
            ax.step(
                bins[:-1],
                bottom_nd,
                where="post",
                color="black",
                lw=1.0,
                ls="--",
                alpha=0.60,
                zorder=5,
                label="no-dip / dip boundary" if (row_idx == 0 and col_idx == 0) else None,
            )

            # ── Pass 2: draw all dipole bars (saturated shades) on top ────
            for cat in cats_present:
                cinfo = CATEGORIES[cat]
                c_sat = cinfo["color"]
                n_d = counts[cat][band]["dip"]

                ax.bar(
                    bin_centres,
                    n_d,
                    width=bar_width,
                    bottom=bottom_d,
                    color=c_sat,
                    edgecolor="white",
                    linewidth=0.3,
                    alpha=0.88,
                    zorder=3,
                    label=(f"{cinfo['label']} dipole" if (row_idx == 0 and col_idx == 0) else None),
                )
                bottom_d += n_d

            # ── Median markers per category (dipole population) ───────────
            for cat in cats_present:
                df_b = data_dict[cat]
                df_bd = df_b[(df_b["r:band"] == band) & df_b["is_dipole"]]
                dmag_d = df_bd["dmag_psf_ap"].dropna().values
                if len(dmag_d) > 2:
                    med = np.median(dmag_d)
                    ax.axvline(
                        med,
                        color=CATEGORIES[cat]["color"],
                        lw=1.6,
                        ls="-.",
                        alpha=0.85,
                        zorder=6,
                    )

            # ── Reference line at Δmag = 0 ────────────────────────────────
            ax.axvline(0.0, color="black", lw=1.0, ls="-", alpha=0.55, zorder=7)

            # ── Formatting ────────────────────────────────────────────────
            ax.set_xlabel(r"$m(\mathrm{psfFlux}) - m(\mathrm{apFlux})$  (mag)", fontsize=9)
            ax.set_ylabel("Counts", fontsize=9)
            ax.set_title(f"band  {band}", color=bcolor, fontweight="bold", fontsize=11)
            ax.set_xlim(DMAG_RANGE)

            # Band-label annotation inside the plot
            ax.text(
                0.97,
                0.97,
                f"band {band}",
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=11,
                fontweight="bold",
                color=bcolor,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=bcolor, alpha=0.8),
            )

    # ── Shared legend (built from first subplot handles) ──────────────────
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=len(cats_present) * 2 + 1,
        fontsize=8,
        framealpha=0.85,
        bbox_to_anchor=(0.5, -0.04),
    )

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    if savename:
        savefig(savename)
    plt.show()


print("plot_fig3_stacked_categories() defined.")

In [ ]:
plot_fig3_stacked_categories(
    data,
    CATS_OK,
    savename="11f_fig3_dmag_psfAp_stacked_categories",
)

## 8. Bonus: per-category breakdown (Figure 1 per category)

Repeat Figure 1 separately for each stellar category to verify that the
distribution shape is consistent across Gaia classification groups and that
the signal is not dominated by a single category.

In [ ]:
for cat in CATS_OK:
    lbl = CATEGORIES[cat]["label"]
    df_c = data[cat]
    print(f"\n=== {lbl} ===")
    plot_fig1_per_band(
        df_c,
        subtitle=lbl,
        savename=f"11f_fig1_dmag_psfAp_per_band_{cat}",
    )

## 8.last plot by category

In [ ]:
def plot_fig4_individual_categories(data_dict, cats, bins=DMAG_BINS, savename=None):
    """Figure 4: 2×3 subplots — one per band —  histogram by category inside each subplot.


    All counts are *raw* (not density-normalised) so the absolute number of
    sources per category is directly readable from bar height.

    Parameters
    ----------
    data_dict : dict  {cat_key: DataFrame}  — one entry per category
    cats      : list  ordered list of category keys to stack
    bins      : array  Δmag bin edges
    savename  : str|None  save PDF+PNG if given
    """
    bin_centres = 0.5 * (bins[:-1] + bins[1:])
    bin_width = np.diff(bins)
    bar_width = bin_width * 0.85  # slight gap between bars

    hist_kw = dict(bins=30, density=True, histtype="stepfilled", edgecolor="white", linewidth=0.4)

    # Pre-compute counts: counts[cat][is_dip] = 1-D array (n_bins,)
    counts = {}
    histval = {}
    for cat in cats:
        if cat not in data_dict:
            continue
        df = data_dict[cat]
        counts[cat] = {}
        histval[cat] = {}
        for band in BANDS:
            df_b = df[df["r:band"] == band]
            dmag_nd = df_b.loc[~df_b["is_dipole"], "dmag_psf_ap"].dropna().values
            dmag_d = df_b.loc[df_b["is_dipole"], "dmag_psf_ap"].dropna().values
            dmag_all = df_b.loc[:, "dmag_psf_ap"].dropna().values
            n_nd, _ = np.histogram(dmag_nd, bins=bins)
            n_d, _ = np.histogram(dmag_d, bins=bins)
            n_a, _ = np.histogram(dmag_all, bins=bins)
            counts[cat][band] = {
                "nodip": n_nd.astype(float),
                "dip": n_d.astype(float),
                "all": n_a.astype(float),
            }
            histval[cat][band] = dmag_all

    cats_present = [c for c in cats if c in counts]

    # ── Build figure ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(5.5 * 3, 5.0 * 2), squeeze=False)
    fig.suptitle(
        r"$\Delta m = m(\mathrm{psfFlux}) - m(\mathrm{apFlux})$",
        #        " — stacked by category\n"
        #        "Light shade = no dipole (bottom)   "
        #        "Saturated shade = dipole (top)   "
        #        "Dashed line = no-dipole / dipole boundary",
        fontsize=12,
        fontweight="bold",
        y=1.02,
    )

    for row_idx, bands_row in enumerate([BANDS_ROW0, BANDS_ROW1]):
        for col_idx, band in enumerate(bands_row):
            ax = axes[row_idx][col_idx]
            bcolor = BAND_COLORS[band]

            # Accumulators for stacking bottoms
            bottom_nd = np.zeros(len(bin_centres))  # bottom of non-dipole stack
            bottom_d = np.zeros(len(bin_centres))  # bottom of dipole stack
            # (starts after all no-dipole bars)

            # ── Pass 1: draw all no-dipole bars (lighter shades) ──────────
            for cat in cats_present:
                cinfo = CATEGORIES[cat]
                c_sat = cinfo["color"]  # saturated colour
                c_lite = lighten_color(c_sat, 0.55)  # lightened colour
                n_nd = counts[cat][band]["nodip"]
                n_a = counts[cat][band]["all"]

                ax.hist(
                    histval[cat][band],
                    bins=bins,
                    color=c_sat,
                    density=True,
                    alpha=0.55,
                    zorder=2,
                    label=f"{cat}",
                )

            # ── Horizontal separator: top of no-dipole stack ──────────────
            # Draw a step-line at the top of the non-dipole stacks so the eye
            # can immediately locate the boundary between the two populations.
            # ax.step(
            #    bins[:-1], bottom_nd,
            #    where="post",
            #    color="black",
            #    lw=1.0,
            #    ls="--",
            #    alpha=0.60,
            #    zorder=5,
            #    label="no-dip / dip boundary" if (row_idx == 0 and col_idx == 0) else None,
            # )

            # ── Pass 2: draw all dipole bars (saturated shades) on top ────
            # for cat in cats_present:
            #    cinfo = CATEGORIES[cat]
            #    c_sat = cinfo["color"]
            #    n_d   = counts[cat][band]["dip"]

            #    ax.bar(
            #        bin_centres,
            #        n_d,
            #        width=bar_width,
            #        bottom=bottom_d,
            #        color=c_sat,
            #        edgecolor="white",
            #        linewidth=0.3,
            #        alpha=0.88,
            #        zorder=3,
            #        label=(f"{cinfo['label']} dipole"
            #               if (row_idx == 0 and col_idx == 0) else None),
            #    )
            #    bottom_d += n_d

            # ── Median markers per category (dipole population) ───────────
            # for cat in cats_present:
            #    df_b   = data_dict[cat]
            #    df_bd  = df_b[(df_b["r:band"] == band) & df_b["is_dipole"]]
            #    dmag_d = df_bd["dmag_psf_ap"].dropna().values
            #    if len(dmag_d) > 2:
            #        med = np.median(dmag_d)
            #        ax.axvline(
            #            med,
            #            color=CATEGORIES[cat]["color"],
            #            lw=1.6,
            #            ls="-.",
            #            alpha=0.85,
            #            zorder=6,
            #        )

            # ── Reference line at Δmag = 0 ────────────────────────────────
            ax.axvline(0.0, color="black", lw=1.0, ls="-", alpha=0.55, zorder=7)

            # ── Formatting ────────────────────────────────────────────────
            ax.set_xlabel(r"$m(\mathrm{psfFlux}) - m(\mathrm{apFlux})$  (mag)", fontsize=9)
            ax.set_ylabel("Counts", fontsize=9)
            ax.set_title(f"band  {band}", color=bcolor, fontweight="bold", fontsize=11)
            ax.set_xlim(DMAG_RANGE)

            # Band-label annotation inside the plot
            ax.text(
                0.97,
                0.97,
                f"band {band}",
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=11,
                fontweight="bold",
                color=bcolor,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=bcolor, alpha=0.8),
            )
            ax.legend()

    # ── Shared legend (built from first subplot handles) ──────────────────
    # fig.legend()
    # handles, labels = axes[0][0].get_legend_handles_labels()
    # fig.legend(
    #    handles, labels,
    #    loc="lower center",
    #    ncol=len(cats_present) * 2 + 1,
    #    fontsize=8,
    #    framealpha=0.85,
    #    bbox_to_anchor=(0.5, -0.04),
    # )

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    if savename:
        savefig(savename)
    plt.show()


print("plot_fig4_individual_categories() defined.")

In [ ]:
plot_fig4_individual_categories(
    data,
    CATS_OK,
    savename="11f_fig4_dmag_psfAp_individual_categories",
)

## 9. Numerical summary table

In [ ]:
rows_sum = []
for cat in CATS_OK:
    df = data[cat]
    lbl = CATEGORIES[cat]["label"]
    for band in BANDS + ["ALL"]:
        df_b = df if band == "ALL" else df[df["r:band"] == band]
        for is_dip, tag in [(False, "no-dipole"), (True, "dipole")]:
            sub = df_b[df_b["is_dipole"] == is_dip]["dmag_psf_ap"].dropna()
            if len(sub) == 0:
                continue
            rows_sum.append(
                {
                    "category": lbl,
                    "band": band,
                    "type": tag,
                    "N": len(sub),
                    "median": round(float(np.median(sub)), 4),
                    "mean": round(float(np.mean(sub)), 4),
                    "std": round(float(np.std(sub)), 4),
                }
            )

df_summary = pd.DataFrame(rows_sum)
print("Δmag = mag(psfFlux) − mag(apFlux)  summary per category × band × dipole flag:")
display(df_summary.style.format({"median": "{:.4f}", "mean": "{:.4f}", "std": "{:.4f}"}))

## 9c

In [ ]:
def plot_fig4_individual_categories(data_dict, cats, bins=DMAG_BINS, savename=None):
    """Figure 4: 2×3 subplots — one per band —  histogram by category inside each subplot.


    All counts are *raw* (not density-normalised) so the absolute number of
    sources per category is directly readable from bar height.

    Parameters
    ----------
    data_dict : dict  {cat_key: DataFrame}  — one entry per category
    cats      : list  ordered list of category keys to stack
    bins      : array  Δmag bin edges
    savename  : str|None  save PDF+PNG if given
    """
    bin_centres = 0.5 * (bins[:-1] + bins[1:])
    bin_width = np.diff(bins)
    bar_width = bin_width * 0.85  # slight gap between bars

    # Pre-compute counts: counts[cat][is_dip] = 1-D array (n_bins,)
    counts = {}
    for cat in cats:
        if cat not in data_dict:
            continue
        df = data_dict[cat]
        counts[cat] = {}
        for band in BANDS:
            df_b = df[df["r:band"] == band]
            dmag_nd = df_b.loc[~df_b["is_dipole"], "dmag_psf_ap"].dropna().values
            dmag_d = df_b.loc[df_b["is_dipole"], "dmag_psf_ap"].dropna().values
            dmag_all = df_b.loc[:, "dmag_psf_ap"].dropna().values
            n_nd, _ = np.histogram(dmag_nd, bins=bins)
            n_d, _ = np.histogram(dmag_d, bins=bins)
            n_a, _ = np.histogram(dmag_all, bins=bins)
            counts[cat][band] = {
                "nodip": n_nd.astype(float),
                "dip": n_d.astype(float),
                "all": n_a.astype(float),
            }

    cats_present = [c for c in cats if c in counts]

    # ── Build figure ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(5.5 * 3, 5.0 * 2), squeeze=False)
    fig.suptitle(
        r"$\Delta m = m(\mathrm{psfFlux}) - m(\mathrm{apFlux})$",
        #        " — stacked by category\n"
        #        "Light shade = no dipole (bottom)   "
        #        "Saturated shade = dipole (top)   "
        #        "Dashed line = no-dipole / dipole boundary",
        fontsize=12,
        fontweight="bold",
        y=1.02,
    )

    for row_idx, bands_row in enumerate([BANDS_ROW0, BANDS_ROW1]):
        for col_idx, band in enumerate(bands_row):
            ax = axes[row_idx][col_idx]
            bcolor = BAND_COLORS[band]

            # Accumulators for stacking bottoms
            bottom_nd = np.zeros(len(bin_centres))  # bottom of non-dipole stack
            bottom_d = np.zeros(len(bin_centres))  # bottom of dipole stack
            # (starts after all no-dipole bars)

            # ── Pass 1: draw all no-dipole bars (lighter shades) ──────────
            for cat in cats_present:
                cinfo = CATEGORIES[cat]
                c_sat = cinfo["color"]  # saturated colour
                c_lite = lighten_color(c_sat, 0.55)  # lightened colour
                n_nd = counts[cat][band]["nodip"]
                n_a = counts[cat][band]["all"]

                ax.bar(
                    bin_centres,
                    # n_nd,
                    n_a,
                    width=bar_width,
                    bottom=bottom_nd,
                    color=c_sat,
                    edgecolor="white",
                    linewidth=0.3,
                    alpha=0.90,
                    zorder=2,
                    # label only once (first band) to avoid legend duplication
                    label=(f"{cinfo['label']}" if (row_idx == 0 and col_idx == 0) else None),
                )
                bottom_nd += n_nd

            # bottom_nd now holds the top of the full non-dipole stack per bin
            # → dipole bars start from here
            bottom_d[:] = bottom_nd

            # ── Horizontal separator: top of no-dipole stack ──────────────
            # Draw a step-line at the top of the non-dipole stacks so the eye
            # can immediately locate the boundary between the two populations.
            # ax.step(
            #    bins[:-1], bottom_nd,
            #    where="post",
            #    color="black",
            #    lw=1.0,
            #    ls="--",
            #    alpha=0.60,
            #    zorder=5,
            #    label="no-dip / dip boundary" if (row_idx == 0 and col_idx == 0) else None,
            # )

            # ── Pass 2: draw all dipole bars (saturated shades) on top ────
            # for cat in cats_present:
            #    cinfo = CATEGORIES[cat]
            #    c_sat = cinfo["color"]
            #    n_d   = counts[cat][band]["dip"]

            #    ax.bar(
            #        bin_centres,
            #        n_d,
            #        width=bar_width,
            #        bottom=bottom_d,
            #        color=c_sat,
            #        edgecolor="white",
            #        linewidth=0.3,
            #        alpha=0.88,
            #        zorder=3,
            #        label=(f"{cinfo['label']} dipole"
            #               if (row_idx == 0 and col_idx == 0) else None),
            #    )
            #    bottom_d += n_d

            # ── Median markers per category (dipole population) ───────────
            # for cat in cats_present:
            #    df_b   = data_dict[cat]
            #    df_bd  = df_b[(df_b["r:band"] == band) & df_b["is_dipole"]]
            #    dmag_d = df_bd["dmag_psf_ap"].dropna().values
            #    if len(dmag_d) > 2:
            #        med = np.median(dmag_d)
            #        ax.axvline(
            #            med,
            #            color=CATEGORIES[cat]["color"],
            #            lw=1.6,
            #            ls="-.",
            #            alpha=0.85,
            #            zorder=6,
            #        )

            # ── Reference line at Δmag = 0 ────────────────────────────────
            ax.axvline(0.0, color="black", lw=1.0, ls="-", alpha=0.55, zorder=7)

            # ── Formatting ────────────────────────────────────────────────
            ax.set_xlabel(r"$m(\mathrm{psfFlux}) - m(\mathrm{apFlux})$  (mag)", fontsize=9)
            ax.set_ylabel("Counts", fontsize=9)
            ax.set_title(f"band  {band}", color=bcolor, fontweight="bold", fontsize=11)
            ax.set_xlim(DMAG_RANGE)

            # Band-label annotation inside the plot
            ax.text(
                0.97,
                0.97,
                f"band {band}",
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=11,
                fontweight="bold",
                color=bcolor,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=bcolor, alpha=0.8),
            )

    # ── Shared legend (built from first subplot handles) ──────────────────
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=len(cats_present) * 2 + 1,
        fontsize=8,
        framealpha=0.85,
        bbox_to_anchor=(0.5, -0.04),
    )

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    if savename:
        savefig(savename)
    plt.show()


print("plot_fig4_individual_categories() defined.")

In [ ]:
plot_fig4_individual_categories(
    data,
    CATS_OK,
    savename="11f_fig3_dmag_psfAp_individual_categories",
)

## 10. Conclusions

### Physical interpretation of the figures

**Figure 1 (per band)**

- If the grey (non-dipole) histogram is narrow and centred on Δmag ≈ 0,
  the PSF-fit and aperture-fit agree well for sources without a dipole artefact,
  confirming the photometric calibration is consistent.
- If the dipole histogram (band colour) is **broader** or **shifted** relative to
  the grey one, the PSF-fit flux on the difference image is systematically different
  from the aperture flux for dipole-flagged sources.  A positive Δmag shift
  (psfFlux > apFlux) would mean the PSF model inflates the flux when a centroid
  offset is present, consistent with the atmospheric-dispersion dipole hypothesis.

**Figure 2 (all bands)**

- Comparing the width of the coloured per-band dipole histograms directly tests
  whether bluer bands (u, g) — where atmospheric dispersion is stronger — show
  a larger PSF–aperture discrepancy than redder bands (z, y).
- A systematic bluer-band-wider trend would strongly support the
  atmospheric-achromatic-dispersion template dipole hypothesis.

**Figure 3 (stacked by category)**

- If the three category stacks (steelblue / seagreen / firebrick) fill each bin
  in **constant proportions**, the Δmag distribution is independent of the Gaia
  classification and the effect is a universal pipeline artefact.
- If the **Gaia variable** segment (firebrick, saturated top) is disproportionately
  large at non-zero Δmag, variable stars contribute more extreme PSF–aperture
  discrepancies, possibly because their variability breaks the template assumption.
- The dash-dot vertical lines show per-category dipole medians; well-separated
  medians would indicate category-dependent systematics.

### Next steps

- Compare the median Δmag of dipole sources vs. airmass (notebook 11e) to check
  whether the shift increases at high airmass, as expected for atmospheric dispersion.
- Cross-check with the dipole fit orientation angle (notebook 11e) to verify
  that the dipole axis is aligned with the parallactic angle direction.

In [ ]:
print("Notebook 11f — mag(psfFlux) − mag(apFlux) dipole diagnostic — complete.")